## BLC data Ingestion Framework
NoteBook: Gold_sql_ingestion_rearc

| Author | Date | Change Record | Description |
| ----------- | ----------- | ----------- | -----------
|Hemant kaushik	|06-Nov-2026|	-	| Notebook for ingesting BLC data from BRNZ to SLVR
 



In [0]:
from pyspark.sql.functions import col,trim,current_timestamp,input_file_name,when,row_number,explode,trim,from_json,avg,stddev_pop,count,sum,min,max,round
from pyspark.sql.window import Window


In [0]:
gold_prefix = "workspace.gold.gold_sql_pr_"
slvr_prefix = "workspace.silver.silver_sql_pr_"

In [0]:

##What are the mean and standard deviation of the annual US population across 2013-2018 inclusive

spark.sql(f"""
CREATE OR REPLACE TABLE
    {gold_prefix}population_stats_2013_2018_sql
USING DELTA
AS
SELECT
    COUNT(*) AS year_count,
    ROUND(AVG(population), 2) AS mean_population,
    ROUND(STDDEV_POP(population), 2) AS population_stddev,
    MIN(year) AS start_year,
    MAX(year) AS end_year
FROM {slvr_prefix}population
WHERE year BETWEEN 2013 AND 2018
  AND nation = 'United States'
""")

# -----------------------------------------------------------
##For every series_id in the BLS data, what's the best year, meaning the year with the largest sum of value across all its quarters?

spark.sql(f"""
CREATE OR REPLACE TABLE
    {gold_prefix}bls_best_year_by_series_sql
USING DELTA
AS
WITH annual_values AS
(
    SELECT
        series_id,
        year,
        SUM(value) AS annual_value_sum
    FROM {slvr_prefix}data
    WHERE period IN ('Q01', 'Q02', 'Q03', 'Q04')
      AND value IS NOT NULL
    GROUP BY
        series_id,
        year
),

ranked_years AS
(
    SELECT
        series_id,
        year,
        annual_value_sum,
        ROW_NUMBER() OVER
        (
            PARTITION BY series_id
            ORDER BY
                annual_value_sum DESC,
                year DESC
        ) AS year_rank
    FROM annual_values
),

series_labels AS
(
    SELECT DISTINCT
        series_id
    FROM {slvr_prefix}series
)

SELECT
    r.series_id,
    r.year AS best_year,
    r.annual_value_sum
FROM ranked_years r
LEFT JOIN series_labels s
    ON r.series_id = s.series_id
WHERE r.year_rank = 1
""")

# -----------------------------------------------------
##For series_id = PRS30006032 and period = Q01, what was the value each year, joined with that year's population where available


spark.sql(f"""
CREATE OR REPLACE TABLE
    {gold_prefix}prs30006032_q01_population_sql
USING DELTA
AS
WITH population_by_year AS
(
    SELECT
        year,
        MAX(population) AS population
    FROM {slvr_prefix}population
    WHERE nation = 'United States'
    GROUP BY year
)

SELECT
    b.series_id,
    b.year,
    b.period,
    b.value,
    p.population
FROM {slvr_prefix}data b
LEFT JOIN population_by_year p
    ON b.year = p.year
WHERE b.series_id = 'PRS30006032'
  AND b.period = 'Q01'
ORDER BY b.year
""")